# بناء تطبيقات توليد الصور (Azure OpenAI)

هناك أكثر من مجرد توليد النصوص في نماذج اللغة الكبيرة. يمكنك أيضًا توليد الصور من أوصاف نصية. الصور كنمط وسيلة مفيدة في مجالات MedTech، والهندسة المعمارية، والسياحة، وتطوير الألعاب، والتسويق، والمزيد. في هذا الدرس، نعمل مع نماذج **GPT Image** الحالية على Microsoft Foundry.

## أهداف التعلم

بحلول نهاية هذا الدرس ستتمكن من:

- شرح ما هو توليد الصور وأين يكون مفيدًا.
- فهم عائلة نماذج `gpt-image` وكيف تختلف عن نماذج DALL·E القديمة.
- بناء تطبيق لتوليد الصور وتحرير الصور.

## ما هو توليد الصور؟

أنماط توليد الصور تنشئ صورًا من نص مُعطى. النماذج الحديثة مثل `gpt-image` تتعلم العلاقة بين النص والصور خلال التدريب، ثم تتحول بشكل تكراري من الضوضاء العشوائية إلى صورة تطابق وصفك النصي.

هناك عائلتان معروفتان من نماذج الصور:

- **`gpt-image` (OpenAI)** — الجيل الحالي المستخدم في هذا الدرس. يدعم توليد الصور من النص وتحرير الصور (الطلاء داخل القناع).
- **Midjourney** — نموذج طرف ثالث شهير مع خدمته الخاصة وسير عمل على Discord.

> نماذج OpenAI القديمة للصورة — **DALL·E 2** و **DALL·E 3** — تعتبر قديمة. DALL·E 3 لم يعد متاحًا للنشر الجديد، والميزات مثل `create_variation` كانت موجودة فقط في DALL·E 2. استخدم نماذج `gpt-image` للتطبيقات الجديدة.

على Microsoft Foundry، **`gpt-image-2`** هو أحدث وأكفأ نموذج للصور وهو الافتراضي الموصى به. كما أن نماذج `gpt-image-1.5` و `gpt-image-1-mini` متاحة أيضًا بشكل عام.

> **مهم:** نماذج `gpt-image` تُرجع الصورة المولدة كـ **base64** (`b64_json`)، وليس كعنوان URL. الكود الخاص بك يقوم بفك ترميز سلسلة base64 إلى بايت ويحفظها — لا يوجد عنوان URL للصورة للتحميل.


## بناء أول تطبيق لتوليد الصور الخاص بك

فما الذي يتطلبه بناء تطبيق لتوليد الصور؟ تحتاج إلى المكتبات التالية:

- **python-dotenv**، يُنصح بشدة باستخدام هذه المكتبة للاحتفاظ بسرّك في ملف *.env* بعيدًا عن الكود.
- **openai**، هذه المكتبة هي التي ستستخدمها للتفاعل مع واجهة برمجة تطبيقات OpenAI.
- **pillow**، للعمل مع الصور في بايثون.

إذا لم تفعل ذلك بالفعل، فاتبع التعليمات في صفحة [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) لإنشاء مورد Microsoft Foundry ونموذج. اختر **gpt-image-2** كنموذج (أحدث نموذج توليد الصور لـ Azure OpenAI؛ DALL·E هو نموذج قديم).

1. أنشئ ملف *.env* بالمحتوى التالي:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    يمكنك العثور على هذه المعلومات في بوابة Microsoft Foundry لموردك في قسم "عمليات النشر".



1. جمع المكتبات أعلاه في ملف يسمى *requirements.txt* على النحو التالي:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. بعد ذلك، قم بإنشاء بيئة افتراضية وتثبيت المكتبات:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> بالنسبة لنظام ويندوز، استخدم الأوامر التالية لإنشاء وتفعيل بيئة العمل الافتراضية الخاصة بك:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. أضف الكود التالي في ملف يسمى *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # استيراد dotenv
    dotenv.load_dotenv()

    # تكوين عميل Azure OpenAI (مايكروسوفت فاوندري).
    # تحتاج نماذج الصور إلى نسخة حديثة من واجهة برمجة التطبيقات — تحقق من وثائق فاوندري لمعرفة النسخة التي يتطلبها نموذجك.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # إنشاء صورة باستخدام واجهة برمجة تطبيقات توليد الصور
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # أدخل نص الطلب هنا
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # مثلاً "gpt-image-2"
        )
        # تعيين المجلد لتخزين الصورة
        image_dir = os.path.join(os.curdir, 'images')

        # إذا لم يكن المجلد موجودًا، فقم بإنشائه
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # تهيئة مسار الصورة (لاحظ أن نوع الملف يجب أن يكون png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # نماذج gpt-image تعيد الصورة بصيغة base64 (b64_json)، وليس كعنوان URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # عرض الصورة في عارض الصور الافتراضي
        image = Image.open(image_path)
        image.show()

    # التقاط الاستثناءات
    except BadRequestError as err:
        print(err)

    ```

دعنا نشرح هذا الكود:

- أولاً، نستورد المكتبات التي نحتاجها، بما في ذلك مكتبة OpenAI ومكتبة dotenv، ووحدة base64، ومكتبة Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- بعد ذلك، نقوم بتحميل متغيرات البيئة من ملف *.env*.

    ```python
    # استيراد dotenv
    dotenv.load_dotenv()
    ```

- بعد ذلك، نهيئ عميل Azure OpenAI (Microsoft Foundry).

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- ثم، نُولّد الصورة:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # أدخل نص الطلب الخاص بك هنا
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    تعيد نماذج `gpt-image` الصورة كسلسلة **base64** في `data[0].b64_json`. نقوم بفك ترميزها إلى بايتات ونكتبها في ملف — لا يوجد عنوان URL للتنزيل.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- وأخيرًا، نفتح الصورة ونستخدم عارض الصور الافتراضي لعرضها:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### المزيد من التفاصيل حول توليد الصورة

لننظر في معلمات `images.generate`:

- **prompt**، هو نص الوصف المستخدم لتوليد الصورة. هنا هو "أرنب على حصان، يحمل مصاصة، في مرج ضبابي حيث تنمو نرجسيات".
- **size**، هو حجم الصورة المولدة (على سبيل المثال `1024x1024`، `1536x1024`، `1024x1536`، أو `"auto"`).
- **n**، هو عدد الصور المولدة. هنا نولد صورة واحدة.
- **model**، هو اسم نشر نموذج الصورة الخاص بك (على سبيل المثال `gpt-image-2`).

> نماذج الصور لا تأخذ معامل `temperature` — هذا تحكم لتوليد النصوص. للحصول على تنوع، استدعِ الـ API مرة أخرى؛ لتقليل التنوع، اجعل الوصف الخاص بك أكثر تحديدًا.

## قدرات إضافية لتوليد الصور

لقد رأيت كيف تولد صورة باستخدام بضع أسطر من بايثون. نماذج `gpt-image` يمكنها أيضًا **تحرير** صورة موجودة. من خلال توفير صورة، وقناع اختياري **mask** (يحدد المنطقة المطلوب تغييرها)، ووصف، يمكنك تغيير جزء من الصورة — على سبيل المثال، إضافة قبعة إلى أرنبنا.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# يتم أيضًا إرجاع التعديلات كـ base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

تحتوي الصورة الأساسية على الأرنب فقط؛ الصورة النهائية تحتوي على القبعة على الأرنب.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
